In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import numpy as np
import jax.numpy as jnp

import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['text.usetex'] = True
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.grid'] = True
mpl.rcParams['axes.xmargin'] = 0
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['legend.frameon'] = False
mpl.rcParams['savefig.bbox'] = 'tight'

import jaxsp as jsp

from scipy import constants as const

import sys
sys.path.insert(0, "/home/joshua/PhD_year_1/jaxsp/Adding_stellar_masses")

import Stellar_sim_funcs as SSF
import importlib
importlib.reload(SSF)


from scipy.interpolate import interp1d

from collections import defaultdict

from jaxsp.constants import h, om, hbar, Msun, GN, c, m22

from collections import defaultdict

from numpy.polynomial.legendre import leggauss
from scipy.special import sph_harm

import s2fft



In [ ]:
m22 = 1
u = jsp.set_schroedinger_units(m22)

G = GN.value * (u.from_cm**3) / (u.from_g * u.from_s**2)

In [ ]:
def Calculating_Phi_from_rho_in_3d_Unit_Test(l, rho_rtp, r, dtheta, dphi, theta, phi, u):

    L = int(l.max()) + 1 # = 24

    G = GN.value * (u.from_cm**3) / (u.from_g * u.from_s**2)

    flm_r = []  # list to hold the spherical harmonic coefficients at each r

    def forward_s2fft(rho_at_r):
        return s2fft.forward(rho_at_r, L, sampling='mw', method='jax')
    
    flm_r = jax.vmap(forward_s2fft)(rho_rtp)  # shape (Nr, l, m)


    f00_r = flm_r[:, 0, L - 1]  # shape (r,) - the l=0, m=0 coeff at each r
    f1n1_r = flm_r[:, 1, L - 2]  # shape (r,) - the l=1, m=-1 coeff at each r
    f11_r = flm_r[:, 1, L]      # shape (r,) - the l=1, m=1 coeff at each r

    fig = plt.figure(figsize = (8,6))
    plt.plot(r * u.to_Kpc , f00_r, label=r'$\rho_{00}$', alpha = 0.6)
    plt.plot(r * u.to_Kpc, f1n1_r, label=r'$\rho_{1-1}$', alpha = 0.6)
    plt.plot(r * u.to_Kpc, -f11_r, label=r'$-\rho_{11}$', alpha = 0.6)
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('r')
    plt.ylabel('Spherical Harmonic Coefficients of rho')
    plt.legend()
    plt.grid()
    plt.show()

    log_f00_r = np.log10(f00_r)
    log_f11_r = np.log10(-f11_r)
    log_f1n1_r = np.log10(f1n1_r)

    slope_00, intercept_00 = np.polyfit(np.log10(r), log_f00_r, 1)

    slope_11, intercept_11 = np.polyfit(np.log10(r), log_f11_r, 1)

    slope_1n1, intercept_1n1 = np.polyfit(np.log10(r), log_f1n1_r, 1)

    print('Slope and intercept of log-log plot of f_00 vs r: ', slope_00, intercept_00)
    print('Slope and intercept of log-log plot of f_11 vs r: ', slope_11, intercept_11)
    print('Slope and intercept of log-log plot of f_1-1 vs r: ', slope_1n1, intercept_1n1)

    #Perform integrals

    # Remember |m| must be less than or equal to l for the rho_lm to be non zero

    #Create (l, m) pairs
    lm_pairs = []
    for l_val in range(L):
        for m_val in range(-l_val, l_val + 1):
            lm_pairs.append((l_val, m_val))

    lm_pairs = jnp.array(lm_pairs)  # shape (N_pairs, 2)

    def compute_phi_for_lm(lm_pair):
        """Compute phi_lm for a single (l, m) pair"""

        l_val = lm_pair[0]
        m_val = lm_pair[1]

        prefix = -4.0 * jnp.pi * G / (2 * l_val + 1)
        m_ind = m_val + L - 1

        f_at_lm = flm_r[:, l_val, m_ind] #(Nr)

        # Integrands
        integrand_ext = r**(1 - l_val) * f_at_lm
        integrand_int = r**(l_val + 2) * f_at_lm

        # Cumulative integration
        dr = jnp.diff(r)

        # Internal integral (from 0 to r)
        avg_int = 0.5 * (integrand_int[1:] + integrand_int[:-1])
        integral_int = jnp.concatenate([
            jnp.array([0.0 + 0.0j]),
            jnp.cumsum(avg_int * dr)
        ])

        # External integral (from r to r_max) - integrate backwards
        integrand_ext_rev = integrand_ext[::-1]
        r_rev = r[::-1]
        dr_rev = jnp.diff(r_rev)
        avg_ext = 0.5 * (integrand_ext_rev[1:] + integrand_ext_rev[:-1])
        integral_ext_rev = jnp.concatenate([
            jnp.array([0.0 + 0.0j]),
            jnp.cumsum(avg_ext * dr_rev)
        ])
        integral_ext = -integral_ext_rev[::-1]

        return prefix * (r**(-(l_val + 1)) * integral_int + r**l_val * integral_ext)

    phi_lm = jax.vmap(compute_phi_for_lm)(lm_pairs)  # shape (N_pairs, Nr)

    # In total_phi, for each l,m there are 1000 values corresponding to r = r_min to r_max

    # We actually want them in the first form with r as the first index and then l, m

    #Reconstruct phi_rlm array using JAX's advanced indexing
    phi_rlm = jnp.zeros((len(r), L, 2*L-1), dtype=complex)

    # Extract l and m indices for all pairs
    l_indices = lm_pairs[:, 0].astype(int)
    m_indices = (lm_pairs[:, 1] + L - 1).astype(int)

    # Use JAX's scatter operation to fill all values at once
    # Transpose phi_lm to shape (Nr, N_pairs) then scatter
    for idx in range(len(lm_pairs)):
        phi_rlm = phi_rlm.at[:, l_indices[idx], m_indices[idx]].set(phi_lm[idx])

    

    phi_00_r = phi_rlm[:, 0, L - 1]  # shape (r,) - the l=0, m=0 coeff at each r
    phi_1n1_r = phi_rlm[:, 1, L - 2]  # shape (r,) - the l=1, m=-1 coeff at each r
    phi_11_r = phi_rlm[:, 1, L]      # shape (r,) - the l=1, m=1 coeff at each r

    fig = plt.figure(figsize = (8,6))
    r_min = r[0]
    r_max = r[-1]


    phi_00_r_func = -(4*np.pi)**(3/2)*G*(r_max - 1/2 * r - 1/2 * r_min**2/r)

    phi_1n1_func = -4*np.pi*G/3 * np.sqrt(2*np.pi/3) * (r*np.log(r_max/r) + 1/3 * r - 1/3*r_min**3/r**2)

    phi_11_func = -phi_1n1_func


    plt.plot(r *u.to_Kpc, phi_00_r, label='phi_00', alpha = 0.6)
    plt.plot(r *u.to_Kpc, phi_1n1_r, label='phi_1-1', alpha = 0.6)
    plt.plot(r *u.to_Kpc, phi_11_r, label='phi_11', alpha = 0.6)

    plt.plot(r *u.to_Kpc, phi_00_r_func, '--', label='Analytic phi_00', alpha = 0.6)
    plt.plot(r *u.to_Kpc, phi_1n1_func, '--', label='Analytic phi_1-1', alpha = 0.6)
    plt.plot(r *u.to_Kpc, phi_11_func, '--', label='Analytic phi_11', alpha = 0.6)
    plt.xscale('log')
    plt.xlabel('r')
    plt.ylabel('Spherical Harmonic Coefficients of Phi')
    plt.grid()
    plt.legend()
    plt.show()

    plt.plot(r*u.to_Kpc, phi_00_r/phi_00_r_func, label='phi_00 ratio')
    plt.plot(r*u.to_Kpc, phi_1n1_r/phi_1n1_func, label='phi_1-1 ratio')
    plt.plot(r*u.to_Kpc, phi_11_r/phi_11_func, label='phi_11 ratio')
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('r')
    plt.ylabel('Numerical / Analytic')
    plt.legend()
    plt.grid()
    plt.show()

    def backward_s2fft(Phi_at_r):
        return s2fft.inverse(Phi_at_r, L, sampling='mw', method='jax')
    
    Phi_rtp = jax.vmap(backward_s2fft)(phi_rlm)  # shape (Nr, n_theta, n_phi)


    # Quadrature weights on the MW equiangular grid
    w_theta = jnp.sin(theta) * dtheta  # (n_theta,)
    w_phi = jnp.ones_like(phi) * dphi      # (n_phi,)

    w = w_theta[:, None] * w_phi[None, :]  # (n_theta, n_phi)
    w = w[None, :, :]

    norm = w.sum()                  # ≈ 4π

    # Angle-averaged radial profile Φ(r)
    Phi_r_dt = jnp.sum(Phi_rtp * w, axis=(1, 2)) / norm  # (Nr,)

    return Phi_rtp, Phi_r_dt, phi_lm



# $\rho(r, \theta, \phi) = \frac{1}{r}(1 + \sin{\theta} \cos{\phi})$

### Take the angle-average value of the function for each r
### $\rho(r) = \int_{0}^{2\pi}d\phi \int_{0}^{\pi} \rho(r, \theta, \phi) * \sin{\theta}d\theta$
### $\therefore \rho(r) = \frac{1}{r}$

In [ ]:
l = np.array([23])

L = int(l.max()) + 1

# McEwen–Wiaux–style equiangular grid
n_theta = L
n_phi   = 2 * L - 1

# Generate theta values
i = np.arange(n_theta)
theta = (np.pi * (2 * i + 1)) / (2 * L - 1)
    
# Generate phi values
j = np.arange(n_phi)
phi = (2 * np.pi * j) / (2 * L - 1)           

Theta, Phi = jnp.meshgrid(theta, phi, indexing="ij")  # both (n_theta, n_phi)

r_min = 0.01 * u.from_Kpc
r_max = 440 * u.from_Kpc


r = np.logspace(np.log10(r_min), np.log10(r_max), 1000)

#Function is rho(r, theta, phi) = 1/r * (1 + sin(theta) * cos(phi))

# Construct ρ(r, θ, φ)

rho_rtp = (1 / r[:, None, None]) * (1 + jnp.sin(Theta)[None, :, :] * jnp.cos(Phi)[None, :, :])  # (Nr, n_theta, n_phi)

print(rho_rtp.shape)

# #Compute angle-averaged radial profile

# Quadrature weights on the MW equiangular grid
dtheta = 2 * jnp.pi / n_phi
dphi   = 2 * jnp.pi / n_phi

w_theta = jnp.sin(theta) * dtheta            # (n_theta,)
w_phi   = jnp.ones_like(phi) * dphi          # (n_phi,)

w = w_theta[:, None] * w_phi[None, :]  # (n_theta, n_phi)
w = w[None, :, :]

norm = w.sum()                      # ≈ 4π

# Angle-averaged radial profile ρ_ψ(r)
# Contract over (θ, φ) with weights, then normalise.
rho_psi_r = jnp.sum(rho_rtp * w, axis=(1, 2)) / norm  # (Nr,)


plt.figure(figsize=(6, 4))
plt.plot(r * u.to_Kpc, rho_psi_r * u.to_Msun / u.to_Kpc**3, label = 'Computed')
plt.plot(r * u.to_Kpc, (1/r) * u.to_Msun / u.to_Kpc**3, '--', label = 'Theoretical function')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(r"$r$")
plt.ylabel(r"$\rho_\psi(r)$")
plt.title("Angle-averaged radial profile")
plt.grid()
plt.legend()
plt.show()

plt.figure(figsize=(8, 6))
plt.plot(r * u.to_Kpc, rho_psi_r * r, label = 'computed / theoretical')
plt.xscale('log')
plt.xlabel(r"$r$")
plt.ylabel(r"$\rho_\mathrm{computed} / \rho_\mathrm{theoretical}$", fontsize = 18)
plt.title("Ratio of angle-averaged radial profiles")
plt.legend()
plt.grid()
plt.show()


plt.figure(figsize=(8, 6))
i = 100
plt.imshow(rho_rtp[i,:,:].real, extent=(0, 2 * np.pi, 0, np.pi), aspect='auto')
plt.colorbar()
plt.xlabel(r'$\phi$')
plt.ylabel(r'$\theta$')
plt.title('Density Distribution at r = %.2f kpc' % (r[i] * u.to_Kpc))
plt.show()



# Computing SHT $\rho(r, \theta, \phi) = \sum_{l,m}\rho_{l,m}(r)Y_{l,m}(\theta, \phi)$

### Only Non zero coefficients are for $Y_{0,0}, Y_{1,1}, Y_{1,-1}$

### From calculations get: 

### $\rho_{00} = \frac{\sqrt{4\pi}}{r}$
### $\rho_{1, -1} = \sqrt{\frac{2\pi}{3}}\frac{1}{r}$
### $\rho_{1, 1} = -\sqrt{\frac{2\pi}{3}}\frac{1}{r}$

### $\therefore$
### $\log{\rho_{0, 0}} = \frac{1}{2}\log{4\pi} - \log{r}$. Intercept = 0.5496
### $\log{\rho_{1, -1}} = \frac{1}{2}\log{\frac{2\pi}{3}} - \log{r}$. Intercept = 0.1605

### Note: the gradients and interepts wont be exact as we are sampling the function at a finite number of $\theta$ and $\phi$ coordinates
### Therefore, also wont just be that $Y_{0,0}$, $Y_{1,-1}$, $Y_{1,1}$ are the only non zero spherical harmonics as the function wont exactly
### equal the linear sum of only these three.


In [ ]:

Phi_rtp, Phi_r_dt, phi_lm = Calculating_Phi_from_rho_in_3d_Unit_Test(l, rho_rtp, r, dtheta, dphi, theta, phi, u)

### Performing SHT, then integration given in http://arxiv.org/abs/2011.13141, then inverse SHT

### $\Phi(r, \theta, \phi) = -4\pi G(R_{max} - \frac{1}{2}r - \frac{1}{2}\frac{R_{min}^2}{r}) -\frac{4\pi G}{3}(r\ln{\frac{R_{max}}{r}} + \frac{1}{3}r - \frac{1}{3}\frac{R_{min}^3}{r^2})\sin{\theta}\cos{\phi}$

### Performing angle average

### $\Phi(r) = -4\pi G(R_{max} - \frac{1}{2}r - \frac{1}{2}\frac{R_{min}^2}{r})$

In [ ]:

Phi_r_maths = -4*np.pi*G*(r_max - 1/2*r - 1/2 * r_min**2 / r)

fig, ax = plt.subplots(figsize=(8,6))
ax.plot(r * u.to_Kpc, Phi_r_dt * u.to_kms**2, label='wavefunction potential at time t = dt from spherical harmonics', lw=2, ls='--')
ax.plot(r * u.to_Kpc, Phi_r_maths * u.to_kms**2, label='analytical potential', lw=2, ls=':')
ax.set_xscale('log')
ax.set_ylabel(r"$V \;\;\mathrm{[km^2 \;s^{-2}]}$", fontsize = 15)
ax.set_xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 15)
plt.legend(fontsize = 14)
plt.show()

plt.plot(r * u.to_Kpc, round(Phi_r_dt / Phi_r_maths, 10), label='Computed / Theoretical')
plt.xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 15)
plt.ylabel(r'$V_{Computed} / V_{Theoretical}$', fontsize = 15)
plt.xscale('log')
#plt.yscale('log')
plt.legend()
plt.show()



R = r[:, None, None]
theta = Theta[None, :, :]
phi = Phi[None, :, :]

Phi_rtp_maths = -4*np.pi*G*(r_max - 1/2*R - 1/2*r_min**2/R) - 4*np.pi*G/3*(R*np.log(r_max/R) + 1/3*R - 1/3*r_min**3/R**2)*np.sin(theta)*np.cos(phi)

plt.figure(figsize=(8, 6))
i = 100
plt.imshow(Phi_rtp[i,:,:].real - Phi_rtp_maths[i,:,:].real, extent=(0, 2 * np.pi, 0, np.pi), aspect='auto')
plt.colorbar()
plt.xlabel(r'$\phi$')
plt.ylabel(r'$\theta$')
plt.title('Potential Distribution at r = %.2f kpc | Code - Theoretical' % (r[i] * u.to_Kpc))
plt.show()

Phi_total_error_sq = (abs((Phi_rtp - Phi_rtp_maths)/Phi_rtp_maths))**2

Phi_total_error_rms = np.sqrt(np.mean(Phi_total_error_sq))

print('Total Phi_rms:', Phi_total_error_rms)



In [ ]:
importlib.reload(SSF)

m22 = 1
u = jsp.set_schroedinger_units(m22)

G = GN.value * (u.from_cm**3) / (u.from_g * u.from_s**2)

l = np.array([23])

L = int(l.max()) + 1

# McEwen–Wiaux–style equiangular grid
n_theta = L
n_phi   = 2 * L - 1

# Generate theta values
i = np.arange(n_theta)
theta = (np.pi * (2 * i + 1)) / (2 * L - 1)

# Generate phi values
j = np.arange(n_phi)
phi = (2 * np.pi * j) / (2 * L - 1)

Theta, Phi = jnp.meshgrid(theta, phi, indexing="ij")  # both (n_theta, n_phi)

r_min = 0.01 * u.from_Kpc
r_max = 440 * u.from_Kpc


r = np.logspace(np.log10(r_min), np.log10(r_max), 1000)

#Function is rho(r, theta, phi) = 1/r * (1 + sin(theta) * cos(phi))

# Construct ρ(r, θ, φ)

rho_rtp = (1 / r[:, None, None]) * (1 + jnp.sin(Theta)[None, :, :] * jnp.cos(Phi)[None, :, :])  # (Nr, n_theta, n_phi)

# #Compute angle-averaged radial profile

# Quadrature weights on the MW equiangular grid
dtheta = 2 * jnp.pi / n_phi
dphi   = 2 * jnp.pi / n_phi

w_theta = jnp.sin(theta) * dtheta            # (n_theta,)
w_phi   = jnp.ones_like(phi) * dphi          # (n_phi,)

w = w_theta[:, None] * w_phi[None, :]  # (n_theta, n_phi)
w = w[None, :, :]

norm = w.sum()                      # ≈ 4π

# Angle-averaged radial profile ρ_ψ(r)
# Contract over (θ, φ) with weights, then normalise.
rho_psi_r = jnp.sum(rho_rtp * w, axis=(1, 2)) / norm  # (Nr,)

Phi_rtp, Phi_r_dt = SSF.Calculating_Phi_from_rho_in_3d_optimized(l, rho_rtp, r, dtheta, dphi, theta, phi)

Phi_rtp = Phi_rtp.real

acc_vec = SSF.Acc_from_pot_3d(Phi_rtp, r, theta, phi)

# Reshape to (r, theta, phi, 3) for a single interpolator
data_transposed = np.moveaxis(acc_vec, 0, -1)  # shape (3, 24, 47, 3)

from scipy.interpolate import RegularGridInterpolator

interp = RegularGridInterpolator(
    (r, theta, phi),
    data_transposed,
    method='linear'
)

# Query - returns shape (3,) vector
result = interp([0.19 * u.from_Kpc, np.pi/2, 0])

print(result)



R = r[:, None, None]
theta = Theta[None, :, :]
phi = Phi[None, :, :]

Phi_rtp_maths = -4*np.pi*G*(r_max - 1/2*R - 1/2*r_min**2/R) - 4*np.pi*G/3*(R*np.log(r_max/R) + 1/3*R - 1/3*r_min**3/R**2)*np.sin(theta)*np.cos(phi)

rms_error = np.sqrt(np.mean(abs(((Phi_rtp-Phi_rtp_maths)/Phi_rtp_maths)**2)))

print(rms_error)




# Testing function to only calc it at r_orbit, r_orbit - 1 and r_orbit + 1

In [ ]:
r_orbit = 0.19 * u.from_Kpc

importlib.reload(SSF)

Phi_rtp, r_vals = SSF.Calculating_Phi_from_rho_in_3d_optimized_at_r_orbit(l, rho_rtp, r, dtheta, dphi, theta, phi, r_orbit)

Phi_rtp = Phi_rtp.real

In [ ]:
print(r_vals * u.to_Kpc)

importlib.reload(SSF)


l = np.array([23])

L = int(l.max()) + 1

# McEwen–Wiaux–style equiangular grid
n_theta = L
n_phi   = 2 * L - 1

# Generate theta values
i = np.arange(n_theta)
theta = (np.pi * (2 * i + 1)) / (2 * L - 1)

# Generate phi values
j = np.arange(n_phi)
phi = (2 * np.pi * j) / (2 * L - 1)


acc_vec = SSF.Acc_from_pot_3d(Phi_rtp, r_vals, theta, phi) # (3, 3, n_theta, n_phi) = (dim, Nr, n_theta, n_phi)


In [ ]:
# Reshape to (r, theta, phi, 3) for a single interpolator
data_transposed = np.moveaxis(acc_vec, 0, -1)  # shape (3, 24, 47, 3)

from scipy.interpolate import RegularGridInterpolator

interp = RegularGridInterpolator(
    (r_vals, theta, phi),
    data_transposed,
    method='linear'
)

# Query - returns shape (3,) vector
result = interp([0.19 * u.from_Kpc, np.pi/2, 0])

print(result)